# Dynamic Multi-Modal Personalization System

internship project: customer-based captions with Qwen3 + Qdrant RAG, product images with Stable Diffusion, and a FastAPI job queue.

**Before running:** In Colab, select **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom. The first model request can take several minutes.

## 1. Sample customers and brand guidelines

In [ ]:
%%writefile customers.json
[
  {
    "customer_id": 101,
    "interests": ["Fitness", "Running"],
    "recently_viewed": ["Running Shoes"]
  },
  {
    "customer_id": 102,
    "interests": ["Fashion", "Lifestyle"],
    "recently_viewed": ["Casual Sneakers"]
  }
]

## 2. Text generation, RAG and image generation

Models are loaded when a request first needs them. Errors use a fallback caption or image.

In [1]:
%%writefile generator.py
"""Caption and image generation for the SportX demo."""

import logging
import time
from pathlib import Path

from PIL import Image, ImageDraw

logger = logging.getLogger("sportx")
BASE_DIR = Path(__file__).resolve().parent
IMAGE_DIR = BASE_DIR / "generated_images"
IMAGE_DIR.mkdir(exist_ok=True)

# Models are loaded only when the first request needs them.
text_model = None
image_model = None
vector_store = None


def get_brand_rules(customer, product):
    global vector_store

    if vector_store is None:
        from langchain_core.documents import Document
        from langchain_huggingface import HuggingFaceEmbeddings
        from langchain_qdrant import QdrantVectorStore
        from langchain_text_splitters import RecursiveCharacterTextSplitter

        brand_text = (BASE_DIR / "brand_guidelines.txt").read_text(encoding="utf-8")
        doc = Document(page_content=brand_text, metadata={"brand": "SportX"})
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
        chunks = splitter.split_documents([doc])
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        vector_store = QdrantVectorStore.from_documents(
            documents=chunks,
            embedding=embeddings,
            location=":memory:",
            collection_name="sportx_brand",
        )

    search_text = (
        f"Guidelines for {product}. "
        f"Interests: {', '.join(customer['interests'])}. "
        f"Viewed: {', '.join(customer['recently_viewed'])}."
    )
    docs = vector_store.similarity_search(search_text, k=3)
    return "\n".join(doc.page_content for doc in docs)


def make_caption(customer, product):
    global text_model
    from transformers import pipeline

    if text_model is None:
        text_model = pipeline(
            "text-generation",
            model="Qwen/Qwen3-1.7B",
            device_map="auto",
            dtype="auto",
        )

    rules = get_brand_rules(customer, product)
    messages = [
        {
            "role": "system",
            "content": (
                "You write short marketing captions for SportX. "
                "Always mention SportX. Do not invent prices, discounts, "
                "features or competing brands. Return only the caption.\n"
                f"Brand guidelines:\n{rules}"
            ),
        },
        {
            "role": "user",
            "content": (
                f"Write one caption for {product}. "
                f"Customer interests: {', '.join(customer['interests'])}. "
                f"Recently viewed: {', '.join(customer['recently_viewed'])}."
            ),
        },
    ]
    response = text_model(
        messages,
        max_new_tokens=80,
        do_sample=False,
        tokenizer_encode_kwargs={"enable_thinking": False},
    )
    caption = response[0]["generated_text"][-1]["content"].strip()

    # A basic check; the prompt alone does not guarantee brand compliance.
    if not caption or "sportx" not in caption.lower():
        raise ValueError("Generated caption did not include the brand name")
    return caption


def make_image_prompt(customer, product):
    interests = " ".join(customer["interests"]).lower()
    if "running" in interests or "fitness" in interests:
        setting = "outdoor running track, active fitness lifestyle"
    else:
        setting = "modern city street, casual fashion lifestyle"

    return (
        f"Realistic commercial photo of {product}, {setting}, "
        "single pair of sneakers, clean composition, natural lighting, "
        "detailed product photography, no text, no logo, no watermark"
    )


def make_image(customer, product, job_id):
    global image_model
    import torch
    from diffusers import StableDiffusionPipeline

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is needed for image generation")

    if image_model is None:
        image_model = StableDiffusionPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-v1-5",
            torch_dtype=torch.float16,
        )
        image_model.enable_model_cpu_offload()

    image = image_model(
        prompt=make_image_prompt(customer, product),
        num_inference_steps=20,
        height=512,
        width=512,
    ).images[0]

    filename = f"marketing_{job_id}.png"
    image.save(IMAGE_DIR / filename)
    return filename


def fallback_image():
    filename = "fallback.png"
    path = IMAGE_DIR / filename
    if not path.exists():
        image = Image.new("RGB", (512, 512), "#f1f5f9")
        draw = ImageDraw.Draw(image)
        draw.text((218, 245), "SportX", fill="#0f172a")
        image.save(path)
    return filename


def generate_content(customer, product, job_id):
    start = time.perf_counter()
    errors = []

    try:
        caption = make_caption(customer, product)
    except Exception:
        logger.exception("Caption generation failed for customer %s", customer["customer_id"])
        caption = f"Explore {product} with SportX."
        errors.append("caption_generation_failed")

    try:
        image_filename = make_image(customer, product, job_id)
    except Exception:
        logger.exception("Image generation failed for customer %s", customer["customer_id"])
        image_filename = fallback_image()
        errors.append("image_generation_failed")

    return {
        "customer_id": customer["customer_id"],
        "product": product,
        "caption": caption,
        "image_url": f"/images/{image_filename}",
        "status": "partial_success" if errors else "success",
        "errors": errors,
        "generation_time_seconds": round(time.perf_counter() - start, 2),
    }

Writing generator.py


## 4. FastAPI backend

One GPU worker processes jobs in order. The queue accepts up to three unfinished jobs.

In [ ]:
%%writefile main.py
"""FastAPI backend for the SportX marketing project."""

import json
import logging
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from uuid import uuid4

from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field, field_validator

from generator import IMAGE_DIR, generate_content

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("sportx")

BASE_DIR = Path(__file__).resolve().parent
customers = json.loads((BASE_DIR / "customers.json").read_text(encoding="utf-8"))

app = FastAPI(title="SportX AI Marketing API", version="1.0")
app.mount("/images", StaticFiles(directory=str(IMAGE_DIR)), name="images")

# One worker prevents several image models from using the same GPU at once.
worker = ThreadPoolExecutor(max_workers=1)
queue_slots = threading.BoundedSemaphore(3)
jobs = {}
jobs_lock = threading.Lock()


class MarketingRequest(BaseModel):
    customer_id: int
    product: str = Field(min_length=1, max_length=100)

    @field_validator("product")
    @classmethod
    def check_product(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("Product cannot be empty")
        return value


@app.get("/")
def home():
    return {"message": "SportX AI Marketing API is running"}


@app.get("/customers")
def list_customers():
    return customers


def release_slot(_future):
    queue_slots.release()


@app.post("/jobs/submit", status_code=202)
def submit_job(request: MarketingRequest):
    customer = next(
        (item for item in customers if item["customer_id"] == request.customer_id),
        None,
    )
    if customer is None:
        raise HTTPException(status_code=404, detail="Customer not found")

    if not queue_slots.acquire(blocking=False):
        raise HTTPException(status_code=429, detail="Queue is full. Try again later.")

    job_id = str(uuid4())
    try:
        future = worker.submit(generate_content, customer, request.product, job_id)
        with jobs_lock:
            jobs[job_id] = future
        future.add_done_callback(release_slot)
    except Exception:
        queue_slots.release()
        logger.exception("Could not submit job")
        raise HTTPException(status_code=500, detail="Could not submit job")

    return {"job_id": job_id, "status": "queued"}


@app.get("/jobs/{job_id}")
def get_job(job_id: str):
    with jobs_lock:
        future = jobs.get(job_id)

    if future is None:
        raise HTTPException(status_code=404, detail="Job not found")

    if not future.done():
        return {"job_id": job_id, "status": "running" if future.running() else "queued"}

    try:
        return {"job_id": job_id, **future.result()}
    except Exception:
        logger.exception("Job %s failed", job_id)
        return {"job_id": job_id, "status": "failed", "errors": ["unexpected_error"]}

## 5. Start the API in Colab

This URL works inside Colab. It is **not** a public deployment link.

In [ ]:
import threading
import time
import uvicorn

from main import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Server thread alive:", server_thread.is_alive())

In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/", timeout=10)
print("HTTP Status:", response.status_code)
print(response.json())

## 6. Generate content through the API

The submit endpoint returns a job ID quickly. Generating the actual image can take much longer than three seconds.

In [ ]:
import requests

base_url = "http://127.0.0.1:8000"
response = requests.post(
    f"{base_url}/jobs/submit",
    json={"customer_id": 101, "product": "Running Shoes"},
    timeout=20,
)
print("HTTP Status:", response.status_code)
print(response.json())
response.raise_for_status()
job_id = response.json()["job_id"]

In [ ]:
import time
from IPython.display import display, Image

for _ in range(60):
    response = requests.get(f"{base_url}/jobs/{job_id}", timeout=15)
    result = response.json()
    print("Status:", result["status"])
    if result["status"] not in ("queued", "running"):
        break
    time.sleep(5)

if result["status"] in ("success", "partial_success"):
    print("Caption:", result["caption"])
    print("Time:", result["generation_time_seconds"], "seconds")
    print("Errors:", result["errors"])
    image_response = requests.get(base_url + result["image_url"], timeout=30)
    image_response.raise_for_status()
    display(Image(data=image_response.content))

## 7. Streamlit frontend

The Streamlit UI is in the GitHub project file `streamlit_app.py`. Deploy it separately from the GPU backend. See `README.md` for GitHub and hosting steps.

**Limitation:** The job ID may be returned quickly, but full Stable Diffusion generation is not guaranteed under three seconds.